In [1]:
import pandas as pd
import numpy as np
import warnings
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Lasso, LassoLars
from sklearn.pipeline import make_pipeline
from sklearn.metrics import f1_score, mean_squared_error, average_precision_score
from sklearn.exceptions import ConvergenceWarning
from joblib import Parallel, delayed
from scipy.stats import binom

warnings.filterwarnings("ignore", category=ConvergenceWarning)

# ---------------------------------------------------------
# Phase 1: Hi-LASSO Procedure I (Importance Scores)
# ---------------------------------------------------------
def generate_importance_scores(b, X_train_scaled, y_train_scaled, q1, p, alpha_val):
    rs = np.random.default_rng(b + 100)
    n = X_train_scaled.shape[0]
    boot_rows = rs.choice(n, size=n, replace=True)
    boot_cols = rs.choice(p, size=q1, replace=False)
    
    # THE FIX: Use LassoLars to properly shrink noise, removing the forced quota!
    model = LassoLars(alpha=alpha_val, max_iter=2000)
    model.fit(X_train_scaled[boot_rows][:, boot_cols], y_train_scaled[boot_rows])
    return boot_cols, model.coef_

# ---------------------------------------------------------
# Phase 2: Hi-LASSO Weighted Refinement Pass
# ---------------------------------------------------------
def adaptive_feature_refinement(b, X_train_scaled, y_train_scaled, q2, p, importances, alpha_val):
    rs = np.random.default_rng(b + 200)
    n = X_train_scaled.shape[0]
    boot_rows = rs.choice(n, size=n, replace=True)
    
    # THE FIX: Exact probability normalization to prevent the NumPy crash
    prob_weights = importances / np.sum(importances)
    prob_weights = prob_weights / np.sum(prob_weights) 
    
    boot_cols = rs.choice(p, size=q2, replace=False, p=prob_weights)
    
    # Strictly Basic LASSO as requested (not adaptive lasso)
    model = LassoLars(alpha=alpha_val, max_iter=2000)
    model.fit(X_train_scaled[boot_rows][:, boot_cols], y_train_scaled[boot_rows])
    return boot_cols, model.coef_

# ---------------------------------------------------------
# Math Helpers
# ---------------------------------------------------------
def calc_rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

def calc_rme(beta_true, beta_hat):
    return np.linalg.norm(beta_hat - beta_true) / (np.linalg.norm(beta_true) + 1e-10)

def calc_rme_nonzeros(beta_true, beta_hat):
    mask = beta_true != 0
    return np.linalg.norm(beta_hat[mask] - beta_true[mask]) / (np.linalg.norm(beta_true[mask]) + 1e-10)

In [5]:
%%time
# =========================================================
# Main Hi-LASSO Execution Pipeline (Real Data - Single Run)
# =========================================================
import pickle
import numpy as np
import pandas as pd
from scipy.stats import pearsonr, binom
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from joblib import Parallel, delayed
from statsmodels.stats.multitest import multipletests

dataset_name = "TCGA_GBM_LGG"
# Use much gentler penalties for real-world data
alphas = [1e-1, 1e-2, 1e-3, 1e-4, 1e-5]

print(f"\nProcessing Real Data for {dataset_name}...")

# 1. Load Real Data from the Pickle file
with open('GBM&LGG_survival_data.pkl', 'rb') as f:
    data = pickle.load(f)

X = np.array(data['X'])
y = np.array(data['y']).ravel()
gene_names = np.array(data['gene'])
p = X.shape[1]

# We need validation data to tune the alpha penalty
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=(1/6), random_state=42)

X_scaler = StandardScaler()
X_train_scaled = X_scaler.fit_transform(X_train)
X_val_scaled = X_scaler.transform(X_val)

y_scaler = StandardScaler()
y_train_scaled = y_scaler.fit_transform(y_train.reshape(-1, 1)).ravel()
y_val_scaled = y_scaler.transform(y_val.reshape(-1, 1)).ravel()

q1, q2 = X_train_scaled.shape[0], X_train_scaled.shape[0]
B_rand = int((p / q1) * 30)

best_h_rmse = np.inf
best_raw_coefs = None
best_s_counts = None  
best_t_counts = None  

# 2. RUN HI-LASSO (Alpha Tuning Loop)
for a in alphas:
    # --- PHASE 1 ---
    p1_results = Parallel(n_jobs=-1, batch_size=100)(
        delayed(generate_importance_scores)(b, X_train_scaled, y_train_scaled, q1, p, a) for b in range(B_rand)
    )

    importances = np.zeros(p)
    times_selected_p1 = np.zeros(p)
    for boot_cols, boot_coefs in p1_results:
        importances[boot_cols] += np.abs(boot_coefs)
        times_selected_p1[boot_cols] += 1

    importances = np.divide(importances, times_selected_p1, out=np.zeros_like(importances), where=times_selected_p1!=0)
    importances[importances == 0] = 1e-10 

    # --- PHASE 2 ---
    p2_results = Parallel(n_jobs=-1, batch_size=100)(
        delayed(adaptive_feature_refinement)(b, X_train_scaled, y_train_scaled, q2, p, importances, a) for b in range(B_rand)
    )

    hilasso_coef_sums = np.zeros(p)
    times_selected_p2 = np.zeros(p) # This acts as 't'
    times_nonzero_p2 = np.zeros(p)  # NEW: This acts as 's'

    for boot_cols, boot_coefs in p2_results:
        hilasso_coef_sums[boot_cols] += boot_coefs
        times_selected_p2[boot_cols] += 1
        
        # NEW: Count how many times the feature survived the LASSO penalty
        non_zero_mask = np.abs(boot_coefs) > 1e-6
        times_nonzero_p2[boot_cols[non_zero_mask]] += 1

    raw_final_coef = np.divide(hilasso_coef_sums, times_selected_p2, out=np.zeros_like(hilasso_coef_sums), where=times_selected_p2!=0)

    # --- HONEST ALPHA TUNING ---
    temp_coef_rmse = raw_final_coef.copy()
    temp_coef_rmse[np.abs(temp_coef_rmse) < 1e-6] = 0.0

    h_pred = X_val_scaled @ temp_coef_rmse
    current_rmse = calc_rmse(y_val_scaled, h_pred)

    if current_rmse < best_h_rmse:
        best_h_rmse = current_rmse
        best_raw_coefs = raw_final_coef.copy()
        best_alpha = a
        # Lock in the 's' and 't' arrays for the winning model
        best_s_counts = times_nonzero_p2.copy()
        best_t_counts = times_selected_p2.copy()

print(f"Optimal Alpha: {best_alpha} | Validation RMSE: {best_h_rmse:.4f}")

# 3. STATISTICAL SIGNIFICANCE (Binomial Test)
total_s = np.sum(best_s_counts)
total_t = np.sum(best_t_counts)
pi = total_s / total_t if total_t > 0 else 0

p_values = np.ones(p)
for i in range(p):
    t_val = best_t_counts[i]
    s_val = best_s_counts[i]
    
    if t_val > 0:
        # We use s_val - 1 to calculate P(X >= s_val) correctly
        p_values[i] = binom.sf(s_val - 1, int(t_val), pi)

# Bonferroni Correction for Multiple Testing
_, pvals_corrected, _, _ = multipletests(p_values, alpha=0.05, method='bonferroni')

# 4. BUILD THE FINAL TABLE 
results = []
for i in range(p):
    coef = best_raw_coefs[i]
    if coef != 0:
        if np.std(X[:, i]) > 0:
            correlation, _ = pearsonr(X[:, i], y)
        else:
            correlation = 0.0
            
        results.append({
            "Gene": gene_names[i],
            "Coefficient": coef,
            "Importance_Score": np.abs(coef),
            "Correlation": correlation,
            "P_Value": p_values[i],
            "Bonferroni_P": pvals_corrected[i],
            "Reference": "" 
        })

# Convert to DataFrame, sort by absolute coefficient (Importance Score), and grab Top 10
df_results = pd.DataFrame(results)

# NEW: Safeguard to check if all features were killed by the penalty
if df_results.empty:
    print("\n[WARNING] All coefficients were shrunk to 0.0! Your alpha penalty is too high.")
else:
    # Sort by absolute coefficient (Importance Score), and grab Top 10
    top_10_df = df_results.sort_values(by="Importance_Score", ascending=False).head(10)

    # Save the finalized clinical data
    top_10_df.to_csv(f"Top_10_Genes_{dataset_name}.csv", index=False)

    print("\n--- Top 10 Discovered Genes ---")
    print(top_10_df[['Gene', 'Coefficient', 'Correlation', 'P_Value', 'Bonferroni_P']].to_string(index=False))


Processing Real Data for TCGA_GBM_LGG...
Optimal Alpha: 0.1 | Validation RMSE: 2.7951

--- Top 10 Discovered Genes ---
     Gene  Coefficient  Correlation       P_Value  Bonferroni_P
  DEFB120     0.115154     0.501925  0.000000e+00  0.000000e+00
  ONECUT3     0.104386     0.476075  0.000000e+00  0.000000e+00
     GSC2     0.091427     0.450987 7.025752e-292 1.389483e-287
  COL22A1    -0.082008    -0.302456  0.000000e+00  0.000000e+00
  UGT1A10     0.070764     0.379763  1.174495e-88  2.322798e-84
     AMTN     0.064322     0.452014 1.121763e-139 2.218510e-135
LINC00442     0.060331     0.449525 3.606451e-146 7.132478e-142
    HILS1    -0.059956    -0.275192  0.000000e+00  0.000000e+00
 USP17L6P     0.057047     0.375904 6.177315e-148 1.221688e-143
    GPR50     0.056356     0.360708 9.110718e-199 1.801827e-194
CPU times: user 48.3 s, sys: 4.17 s, total: 52.5 s
Wall time: 2min 2s
